In [55]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

In [56]:
online_retail = fetch_ucirepo(id=352)

# data (as pandas dataframes)
X = online_retail.data.features 
y = online_retail.data.targets

In [57]:
ids = online_retail.data.ids
ids

,InvoiceNo,StockCode
0,536365,85123A
1,536365,71053
2,536365,84406B
3,536365,84029G
4,536365,84029E
...,...,...
541904,581587,22613
541905,581587,22899
541906,581587,23254
541907,581587,23255


In [58]:
df = ids.join(X)
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12/9/2011 12:50,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/2011 12:50,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/2011 12:50,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/2011 12:50,4.15,12680.0,France


In [59]:
df['InvoiceDate'].str.split()

0          [12/1/2010, 8:26]
1          [12/1/2010, 8:26]
2          [12/1/2010, 8:26]
3          [12/1/2010, 8:26]
4          [12/1/2010, 8:26]
                 ...        
541904    [12/9/2011, 12:50]
541905    [12/9/2011, 12:50]
541906    [12/9/2011, 12:50]
541907    [12/9/2011, 12:50]
541908    [12/9/2011, 12:50]
Name: InvoiceDate, Length: 541909, dtype: object

### Understanding Missing Value

In [60]:
missing_values = df.isnull().sum()
print(missing_values)

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


### Handling Missing Value

In [61]:
## Drop Rows with Crucial Missing Data
df = df.dropna(subset=['CustomerID'])

## Fill Missing Descriptions (Fixing FutureWarning)
df['Description'] = df['Description'].fillna('Unknown Product')

## Check Duplicates
print(f"Duplicate Rows: {df.duplicated().sum()}")

## Remove Duplicates
df = df.drop_duplicates().reset_index(drop=True)

## Handling Incorrect or Invalid Data
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

## Convert Data Types for Consistency
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

Duplicate Rows: 5225


In [47]:
df['Amount'] = df['UnitPrice'] * X['Quantity']
df['Date']= df['InvoiceDate'].transform(lambda x: x.split()[0])
df['Time'] = df['InvoiceDate'].transform(lambda x: x.split()[-1])

In [48]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
latest_date = df['InvoiceDate'].max()
df['Recency'] = (latest_date - df.groupby('CustomerID')['InvoiceDate'].transform('max')).dt.days


In [49]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Amount,Date,Time,Recency
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,12/1/2010,8:26,301.0
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,12/1/2010,8:26,301.0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,12/1/2010,8:26,301.0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,12/1/2010,8:26,301.0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,12/1/2010,8:26,301.0
...,...,...,...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France,10.20,12/9/2011,12:50,0.0
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,12.60,12/9/2011,12:50,0.0
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,16.60,12/9/2011,12:50,0.0
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,16.60,12/9/2011,12:50,0.0


In [52]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID,Amount,Recency
count,541909.000000,541909,541909.000000,406829.000000,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570,17.987795,38.954140
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000,-168469.600000,0.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000,3.400000,3.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000,9.750000,15.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000,17.400000,40.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000,168469.600000,373.000000
std,218.081158,NaN,96.759853,1713.600303,378.810824,64.686233


In [63]:
CodeTypes = list(map(lambda codes: any(char.isdigit() for char in codes), df['StockCode']))
IrrelevantCodes = [i for i, v in enumerate(CodeTypes) if v == False]
IrrelevantCodes

[45,
 377,
 1087,
 1628,
 1639,
 3121,
 3719,
 3893,
 3955,
 3999,
 4330,
 4404,
 4523,
 4552,
 4718,
 4781,
 4818,
 5460,
 5483,
 6020,
 6077,
 6078,
 6742,
 6746,
 6766,
 6877,
 7003,
 7155,
 7514,
 7712,
 7734,
 8575,
 8811,
 8923,
 9381,
 9577,
 9664,
 9817,
 10597,
 10831,
 11626,
 11674,
 11676,
 11807,
 12605,
 12628,
 12648,
 12721,
 12784,
 12798,
 12838,
 12895,
 13644,
 13874,
 14241,
 14395,
 15159,
 16985,
 17229,
 18182,
 18233,
 18624,
 18643,
 18650,
 20460,
 21012,
 22544,
 22574,
 22741,
 22761,
 22858,
 22903,
 22930,
 22951,
 22997,
 23004,
 23019,
 23170,
 23284,
 23302,
 23534,
 23982,
 24116,
 24395,
 24494,
 24849,
 24923,
 25059,
 25326,
 25425,
 25497,
 25639,
 25919,
 26185,
 26219,
 26907,
 27152,
 27186,
 27826,
 28122,
 28394,
 28807,
 28880,
 28996,
 29160,
 29208,
 29298,
 29669,
 30233,
 30837,
 30979,
 31034,
 31037,
 31337,
 31650,
 31886,
 31926,
 32066,
 32282,
 32396,
 32466,
 32660,
 33079,
 33105,
 33180,
 33848,
 34054,
 34148,
 34177,
 34215,
 